# Solutions — NYC Taxi Analytics (Medium 01)

**Dataset:** `samples.nyctaxi.trips`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

trips = spark.table("samples.nyctaxi.trips")

## Solution 1 — Trip Duration

In [ ]:
# Why: (dropoff - pickup) in seconds / 60 gives minutes; filter keeps valid trips only
result_1 = (
    trips
    .withColumn("duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .filter(F.col("duration_minutes") > 0)
    .select("tpep_pickup_datetime","tpep_dropoff_datetime","trip_distance","fare_amount","duration_minutes")
)

## Solution 2 — Trips & Revenue by Hour

In [ ]:
# Why: F.hour extracts 0-23; sort by hour gives readable time series
result_2 = (
    trips
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    )
    .orderBy("pickup_hour")
)

## Solution 3 — Fare per Mile

In [ ]:
# Why: filter distance > 0 avoids division by zero; round keeps output clean
result_3 = (
    trips
    .filter(F.col("trip_distance") > 0)
    .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2))
    .select("pickup_zip","dropoff_zip","trip_distance","fare_amount","fare_per_mile")
)

## Solution 4 — Top 10 Pickup-Dropoff Pairs

In [ ]:
# Why: groupBy pair then limit 10 after sorting
result_4 = (
    trips
    .groupBy("pickup_zip","dropoff_zip")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.col("trip_count").desc())
    .limit(10)
)

## Solution 5 — Top 3 Pickup Zips per Dropoff Zip

In [ ]:
# Why: Window.partitionBy dropoff_zip lets rank restart per dropoff zone
w = Window.partitionBy("dropoff_zip").orderBy(F.col("trip_count").desc())
result_5 = (
    trips
    .groupBy("dropoff_zip","pickup_zip")
    .agg(F.count("*").alias("trip_count"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 3)
    .orderBy("dropoff_zip","rank")
)

## Solution 6 — Avg Fare & Distance per Zip (>100 trips)

In [ ]:
# Why: having filter equivalent done with DataFrame .filter after agg
result_6 = (
    trips
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    )
    .filter(F.col("trip_count") > 100)
    .orderBy(F.col("trip_count").desc())
)

## Solution 7 — Running Total of Fare

In [ ]:
# Why: rowsBetween(unboundedPreceding, currentRow) is the classic cumulative sum frame
w = Window.orderBy("tpep_pickup_datetime").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_7 = (
    trips
    .select(
        "tpep_pickup_datetime",
        "fare_amount",
        F.round(F.sum("fare_amount").over(w), 2).alias("running_total_fare")
    )
)